In [1]:
"""
=============================================================================
Phase 3: Asymmetric Quantile Refinement  (SELF-CONTAINED — no imports
from resgru_unet_pipeline needed)
=============================================================================
Loads the 0.8768 best_episodic_finetune.pt checkpoint and fine-tunes with
a Huber-Pinball hybrid loss targeting the 75th percentile of the pollution
distribution — mathematically penalising under-predictions of episodes 3x.

Key changes vs. Phase 2:
  - Loss: 0.4 * HuberLoss(log-space) + 0.6 * Pinball(q=0.75, log-space)
  - LR: 2e-5  (tilting weights, not re-training)
  - Epochs: 5
  - Inference: Temporal Persistence Calibration (+0.5 % per hour, t=0..15)
=============================================================================
"""

import os
import gc
import time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler


# ============================================================================
# CONFIGURATION  (architecture params must match checkpoint exactly)
# ============================================================================

class Config:
    IS_KAGGLE = os.path.exists('/kaggle/input')
    if IS_KAGGLE:
        DATA_ROOT   = '/kaggle/input/competitions/anrf-aise-hack-phase-2-theme-2-pollution-forecasting-iitd/aisehack-theme-2'
        WORKING_DIR = '/kaggle/working'
    else:
        DATA_ROOT   = './aisehack-theme-2'
        WORKING_DIR = '.'

    RAW_DIR  = os.path.join(DATA_ROOT, 'raw')
    TEST_DIR = os.path.join(DATA_ROOT, 'test_in')

    MONTHS = ['APRIL_16', 'JULY_16', 'OCT_16', 'DEC_16']

    FEATURES = [
        'cpm25', 'q2', 't2', 'u10', 'v10', 'pblh',
        'psfc', 'swdown', 'rain', 'PM25', 'NOx', 'SO2', 'NH3',
    ]
    NUM_FEATURES          = len(FEATURES)
    NUM_INPUT_CHANNELS    = NUM_FEATURES + 3   # +wind_speed, hour_sin, hour_cos
    MAX_SCALE_FEATURES    = {'PM25', 'NH3', 'SO2', 'NOx', 'NMVOC_e', 'NMVOC_finn', 'bio'}
    LOG_TRANSFORM_FEATURES = {'rain'}

    INPUT_HOURS  = 10
    OUTPUT_HOURS = 16
    WINDOW_SIZE  = INPUT_HOURS + OUTPUT_HOURS   # 26

    H = 140
    W = 124

    # Architecture (must match checkpoint)
    ENC_CHANNELS = [64, 128, 256]
    GRU_HIDDEN   = 256
    DEC_CHANNELS = [128, 64, 32]
    DROP_RATE    = 0.1

    # Episode detection thresholds (used for monitoring only, not in loss)
    EPISODE_STD_THRESHOLD = 2.0
    EPISODE_MIN_CPM       = 1.0

    # Inference
    INFERENCE_BATCH_SIZE = 4
    NUM_TEST_SAMPLES     = 218

    SEED = 42


class FinetuneConfig:
    # ---- Phase 3 quantile settings ----
    STRIDE       = 1
    BATCH_SIZE   = 4
    VAL_SPLIT    = 0.1
    EPOCHS       = 5
    LR           = 2e-5     # Ultra-low — tilting, not rebuilding
    WEIGHT_DECAY = 1e-4

    # Pinball loss hyperparameters
    QUANTILE      = 0.75   # Target 75th percentile
    HUBER_DELTA   = 1.0
    HUBER_WEIGHT  = 0.4
    PINBALL_WEIGHT = 0.6

    # Temporal Persistence Calibration
    TPC_RATE = 0.005        # +0.5 % boost per output hour

    EPS = 1e-8

    WORKING_DIR    = Config.WORKING_DIR
    QUANTILE_CKPT  = os.path.join(Config.WORKING_DIR, 'best_quantile_finetune.pt')

    # Phase 2 output is our starting point (0.8768)
    BASE_CKPT_PATH   = '/kaggle/input/notebooks/kaori02/res-1/best_resgru_unet.pt'
    PHASE2_CKPT_PATH = '/kaggle/input/notebooks/kaori02/res-1fine/best_episodic_finetune.pt'
    NORM_STATS_PATH  = '/kaggle/input/notebooks/kaori02/res-1/norm_stats_resgru.npz'

    @staticmethod
    def find_file(filename: str) -> str:
        hardcoded = {
            'best_resgru_unet.pt':        FinetuneConfig.BASE_CKPT_PATH,
            'best_episodic_finetune.pt':  FinetuneConfig.PHASE2_CKPT_PATH,
            'norm_stats_resgru.npz':      FinetuneConfig.NORM_STATS_PATH,
        }
        if filename in hardcoded:
            return hardcoded[filename]
        return os.path.join(Config.WORKING_DIR, filename)


def seed_everything(seed: int = Config.SEED):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = True


# ============================================================================
# DATA PIPELINE  (identical to base pipeline)
# ============================================================================

class NormStats:
    def __init__(self):
        self.mean = {}
        self.std  = {}

    def compute_from_raw(self, raw_dir, months, features):
        print("Computing normalization statistics...")
        for feat in features:
            if feat in Config.MAX_SCALE_FEATURES:
                global_max = 0.0
                for month in months:
                    data = np.load(os.path.join(raw_dir, month, f'{feat}.npy'), mmap_mode='r')
                    global_max = max(global_max, float(np.max(np.abs(data))))
                self.mean[feat] = np.zeros((Config.H, Config.W), dtype=np.float32)
                self.std[feat]  = np.full((Config.H, Config.W), max(global_max, 1e-15), dtype=np.float32)

            elif feat in Config.LOG_TRANSFORM_FEATURES:
                s, sq, n = 0.0, 0.0, 0
                for month in months:
                    data = np.load(os.path.join(raw_dir, month, f'{feat}.npy'), mmap_mode='r')
                    for t in range(data.shape[0]):
                        x = np.log1p(data[t].astype(np.float64))
                        s += x.sum(); sq += (x**2).sum(); n += x.size
                mean_ = s / n
                std_  = max(np.sqrt(sq / n - mean_**2), 1e-6)
                self.mean[feat] = np.full((Config.H, Config.W), mean_, dtype=np.float32)
                self.std[feat]  = np.full((Config.H, Config.W), std_,  dtype=np.float32)

            else:
                cnt, m_acc, m2 = 0, np.zeros((Config.H, Config.W), np.float64), np.zeros((Config.H, Config.W), np.float64)
                for month in months:
                    data = np.load(os.path.join(raw_dir, month, f'{feat}.npy'), mmap_mode='r')
                    for t in range(data.shape[0]):
                        x = data[t].astype(np.float64); cnt += 1
                        d = x - m_acc; m_acc += d / cnt; m2 += d * (x - m_acc)
                self.mean[feat] = m_acc.astype(np.float32)
                self.std[feat]  = np.maximum(np.sqrt(m2 / max(cnt-1,1)).astype(np.float32), 1e-6)
            print(f"  {feat}: done")

    def normalize(self, data, feat_name):
        if feat_name in Config.LOG_TRANSFORM_FEATURES:
            data = np.log1p(data)
        return (data - self.mean[feat_name]) / self.std[feat_name]

    def save(self, path):
        np.savez(path, features=list(self.mean.keys()),
                 **{f'mean_{k}': v for k, v in self.mean.items()},
                 **{f'std_{k}':  v for k, v in self.std.items()})

    def load(self, path):
        d = np.load(path)
        for feat in list(d['features']):
            self.mean[feat] = d[f'mean_{feat}']
            self.std[feat]  = d[f'std_{feat}']


def compute_engineered_features(u10, v10, hour_indices):
    wind_speed = np.sqrt(u10**2 + v10**2).astype(np.float32)
    T, H, W    = u10.shape
    hours      = hour_indices.astype(np.float32)
    hour_sin   = np.sin(2 * np.pi * hours / 24.0)[:, None, None] * np.ones((1,H,W), np.float32)
    hour_cos   = np.cos(2 * np.pi * hours / 24.0)[:, None, None] * np.ones((1,H,W), np.float32)
    return wind_speed, hour_sin, hour_cos


class PM25TrainDataset(Dataset):
    def __init__(self, raw_dir, months, features, norm_stats, stride=1):
        self.features   = features
        self.norm_stats = norm_stats
        self.samples    = []
        self.data_paths = {}

        for m_idx, month in enumerate(months):
            T = np.load(os.path.join(raw_dir, month, 'cpm25.npy'), mmap_mode='r').shape[0]
            n = (T - Config.WINDOW_SIZE) // stride + 1
            for i in range(n):
                self.samples.append((m_idx, i * stride))
            for feat in features:
                self.data_paths[(m_idx, feat)] = os.path.join(raw_dir, month, f'{feat}.npy')

        print(f"  TrainDataset: {len(self.samples)} samples, stride={stride}")

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        m_idx, start = self.samples[idx]
        input_list, u10_norm, v10_norm = [], None, None

        for feat in self.features:
            chunk = np.load(self.data_paths[(m_idx, feat)], mmap_mode='r')[start:start+Config.INPUT_HOURS].astype(np.float32)
            chunk = self.norm_stats.normalize(chunk, feat)
            input_list.append(chunk)
            if feat == 'u10': u10_norm = chunk
            elif feat == 'v10': v10_norm = chunk

        ws, hs, hc = compute_engineered_features(u10_norm, v10_norm, np.arange(start, start+Config.INPUT_HOURS) % 24)
        input_list.extend([ws, hs, hc])
        inputs = np.stack(input_list, axis=0)   # (C, 10, H, W)

        cpm_raw    = np.load(self.data_paths[(m_idx, 'cpm25')], mmap_mode='r')
        target_raw = cpm_raw[start+Config.INPUT_HOURS : start+Config.WINDOW_SIZE].astype(np.float32)
        target_log = np.log1p(target_raw)
        last_log   = np.log1p(cpm_raw[start+Config.INPUT_HOURS-1].astype(np.float32))
        ctx_raw    = cpm_raw[start:start+Config.INPUT_HOURS].astype(np.float32)

        return (torch.from_numpy(inputs),
                torch.from_numpy(target_log),
                torch.from_numpy(target_raw),
                torch.from_numpy(last_log),
                torch.from_numpy(ctx_raw))


class PM25TestDataset(Dataset):
    def __init__(self, test_dir, features, norm_stats):
        self.features   = features
        self.norm_stats = norm_stats
        self.data       = {f: np.load(os.path.join(test_dir, f'{f}.npy'), mmap_mode='c') for f in features}
        self.n          = self.data[features[0]].shape[0]
        print(f"  TestDataset: {self.n} samples")

    def __len__(self): return self.n

    def __getitem__(self, idx):
        input_list, u10_norm, v10_norm = [], None, None
        for feat in self.features:
            chunk = self.data[feat][idx].astype(np.float32)
            chunk = self.norm_stats.normalize(chunk, feat)
            input_list.append(chunk)
            if feat == 'u10': u10_norm = chunk
            elif feat == 'v10': v10_norm = chunk

        ws, hs, hc = compute_engineered_features(u10_norm, v10_norm,
                                                  np.arange(Config.INPUT_HOURS).astype(np.float32))
        input_list.extend([ws, hs, hc])
        inputs   = np.stack(input_list, axis=0)
        last_log = np.log1p(self.data['cpm25'][idx].astype(np.float32)[-1])
        return torch.from_numpy(inputs), torch.from_numpy(last_log)


# ============================================================================
# MODEL ARCHITECTURE  (identical to base — must match checkpoint weights)
# ============================================================================

class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, drop_rate=0.0):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False)
        self.gn1   = nn.GroupNorm(min(32, out_ch), out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False)
        self.gn2   = nn.GroupNorm(min(32, out_ch), out_ch)
        self.drop  = nn.Dropout2d(drop_rate) if drop_rate > 0 else nn.Identity()
        self.skip  = nn.Conv2d(in_ch, out_ch, 1, bias=False) if in_ch != out_ch else nn.Identity()

    def forward(self, x):
        r = self.skip(x)
        x = F.gelu(self.gn1(self.conv1(x)))
        x = self.drop(x)
        x = F.gelu(self.gn2(self.conv2(x)))
        return x + r


class ConvGRUCell(nn.Module):
    def __init__(self, input_dim, hidden_dim, kernel_size=3):
        super().__init__()
        p = kernel_size // 2
        self.hidden_dim     = hidden_dim
        self.conv_gates     = nn.Conv2d(input_dim+hidden_dim, 2*hidden_dim, kernel_size, padding=p, bias=True)
        self.conv_candidate = nn.Conv2d(input_dim+hidden_dim,   hidden_dim, kernel_size, padding=p, bias=True)
        self.gn_gates       = nn.GroupNorm(min(32, 2*hidden_dim), 2*hidden_dim)
        self.gn_cand        = nn.GroupNorm(min(32,   hidden_dim),   hidden_dim)

    def forward(self, x, h):
        comb  = torch.cat([x, h], dim=1)
        gates = torch.sigmoid(self.gn_gates(self.conv_gates(comb)))
        r, u  = gates.chunk(2, dim=1)
        cand  = torch.tanh(self.gn_cand(self.conv_candidate(torch.cat([x, r*h], dim=1))))
        return (1-u)*h + u*cand


class ResGRUUNet(nn.Module):
    def __init__(self, cfg=Config):
        super().__init__()
        self.cfg = cfg
        C_in   = cfg.NUM_INPUT_CHANNELS * cfg.INPUT_HOURS
        enc_ch = cfg.ENC_CHANNELS
        dec_ch = cfg.DEC_CHANNELS
        drop   = cfg.DROP_RATE

        self.enc0  = ConvBlock(C_in,      enc_ch[0], drop)
        self.pool0 = nn.MaxPool2d(2)
        self.enc1  = ConvBlock(enc_ch[0], enc_ch[1], drop)
        self.pool1 = nn.MaxPool2d(2)
        self.enc2  = ConvBlock(enc_ch[1], enc_ch[2], drop)
        self.pool2 = nn.AvgPool2d(2, ceil_mode=True)

        self.bottleneck = ConvBlock(enc_ch[2], cfg.GRU_HIDDEN, drop)
        self.convgru    = ConvGRUCell(cfg.GRU_HIDDEN, cfg.GRU_HIDDEN, kernel_size=3)
        self.gru_proj   = nn.Conv2d(cfg.GRU_HIDDEN, enc_ch[2], 1, bias=False)

        self.up2  = nn.ConvTranspose2d(enc_ch[2], dec_ch[0], 2, stride=2)
        self.dec2 = ConvBlock(dec_ch[0]+enc_ch[2], dec_ch[0], drop)
        self.up1  = nn.ConvTranspose2d(dec_ch[0], dec_ch[1], 2, stride=2)
        self.dec1 = ConvBlock(dec_ch[1]+enc_ch[1], dec_ch[1], drop)
        self.up0  = nn.ConvTranspose2d(dec_ch[1], dec_ch[2], 2, stride=2)
        self.dec0 = ConvBlock(dec_ch[2]+enc_ch[0], dec_ch[2], drop)

        self.head = nn.Sequential(
            nn.Conv2d(dec_ch[2], dec_ch[2], 3, padding=1, bias=False),
            nn.GroupNorm(min(32, dec_ch[2]), dec_ch[2]),
            nn.GELU(),
            nn.Conv2d(dec_ch[2], 1, 1),
        )
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d)):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None: nn.init.zeros_(m.bias)
            elif isinstance(m, nn.GroupNorm):
                nn.init.ones_(m.weight); nn.init.zeros_(m.bias)
        nn.init.zeros_(self.head[-1].weight)
        nn.init.zeros_(self.head[-1].bias)

    def _pad(self, x, target):
        dh = target.shape[2] - x.shape[2]
        dw = target.shape[3] - x.shape[3]
        if dh > 0 or dw > 0: return F.pad(x, [0, dw, 0, dh])
        if dh < 0 or dw < 0: return x[:, :, :target.shape[2], :target.shape[3]]
        return x

    def forward(self, x, last_frame_log):
        B, C, T, H, W = x.shape
        x  = x.reshape(B, C*T, H, W)
        e0 = self.enc0(x)
        e1 = self.enc1(self.pool0(e0))
        e2 = self.enc2(self.pool1(e1))
        bn = self.bottleneck(self.pool2(e2))
        h  = bn
        outputs = []
        for _ in range(self.cfg.OUTPUT_HOURS):
            h   = self.convgru(bn, h)
            d   = self.gru_proj(h)
            d2  = self.dec2(torch.cat([self._pad(self.up2(d),  e2), e2], dim=1))
            d1  = self.dec1(torch.cat([self._pad(self.up1(d2), e1), e1], dim=1))
            d0  = self.dec0(torch.cat([self._pad(self.up0(d1), e0), e0], dim=1))
            outputs.append(self.head(d0).squeeze(1))
        deltas = torch.stack(outputs, dim=1)
        return last_frame_log.unsqueeze(1) + deltas


# ============================================================================
# HUBER-PINBALL HYBRID LOSS  — targets the 75th percentile
# ============================================================================

class HuberPinballLoss(nn.Module):
    """
    Total_Loss = 0.4 * HuberLoss(pred_log, target_log)
               + 0.6 * PinballLoss(pred_log, target_log, q=0.75)

    Operating in log-space keeps gradients stable.  The pinball term
    penalises under-predictions 3× more than over-predictions (since
    q / (1-q) = 0.75 / 0.25 = 3), forcing the model to chase the
    high-intensity tail of the pollution distribution.
    """

    def __init__(self):
        super().__init__()
        self.q      = FinetuneConfig.QUANTILE        # 0.75
        self.delta  = FinetuneConfig.HUBER_DELTA     # 1.0
        self.w_hub  = FinetuneConfig.HUBER_WEIGHT    # 0.4
        self.w_pin  = FinetuneConfig.PINBALL_WEIGHT  # 0.6
        self.eps    = FinetuneConfig.EPS

    def forward(self, pred_log, target_log, target_raw, last_frame_log, context_cpm_raw):
        # ---- Huber loss (log-space) ----
        base_loss = F.huber_loss(pred_log, target_log, delta=self.delta, reduction='mean')

        # ---- Pinball / quantile loss (log-space) ----
        # residual = y_true - y_pred   (positive → under-prediction)
        residual     = target_log - pred_log
        pinball_loss = torch.mean(torch.max(
            self.q       * residual,
            (self.q - 1) * residual,
        ))

        total_loss = self.w_hub * base_loss + self.w_pin * pinball_loss

        # ---- Monitoring in raw space (no grad) ----
        with torch.no_grad():
            pred_raw = F.relu(torch.expm1(pred_log.clamp(max=15)))

            # Episode mask — for SMAPE reporting only
            b_mean = context_cpm_raw.mean(dim=1, keepdim=True)
            b_std  = context_cpm_raw.std(dim=1,  keepdim=True) + self.eps
            is_ep  = (
                (target_raw > b_mean + Config.EPISODE_STD_THRESHOLD * b_std)
                & (target_raw > Config.EPISODE_MIN_CPM)
            )

            numer  = torch.abs(pred_raw - target_raw)
            denom  = (torch.abs(pred_raw) + torch.abs(target_raw)) * 0.5 + self.eps
            smape  = (numer / denom).mean().item()

            ep_f     = is_ep.float()
            n_ep     = ep_f.sum().item()
            ep_smape = ((numer / denom) * ep_f).sum().item() / max(n_ep, 1.0)

            ep_corr = 0.0
            if n_ep >= 10:
                flat = is_ep.reshape(-1)
                p_e  = pred_raw.reshape(-1)[flat]
                y_e  = target_raw.reshape(-1)[flat]
                pc   = p_e - p_e.mean();  yc = y_e - y_e.mean()
                cov  = (pc * yc).sum()
                ep_corr = (cov / (torch.sqrt((pc**2).sum() + self.eps) *
                                  torch.sqrt((yc**2).sum() + self.eps))).item()

        return total_loss, {
            'global_smape':  smape,
            'episode_smape': ep_smape,
            'episode_corr':  ep_corr,
            'n_episodes':    n_ep,
            'base_loss':     base_loss.item(),
            'pinball_loss':  pinball_loss.item(),
        }


# ============================================================================
# TRAINING UTILITIES
# ============================================================================

@torch.no_grad()
def validate_quantile(model, loader, criterion, device):
    model.eval()
    total, metrics_acc, n = 0.0, {}, 0
    for inputs, t_log, t_raw, lf_log, ctx in loader:
        inputs, t_log, t_raw, lf_log, ctx = (
            inputs.to(device, non_blocking=True),
            t_log.to(device, non_blocking=True),
            t_raw.to(device, non_blocking=True),
            lf_log.to(device, non_blocking=True),
            ctx.to(device, non_blocking=True),
        )
        with autocast(dtype=torch.float16):
            pred_log = model(inputs, lf_log)
            loss, metrics = criterion(pred_log.float(), t_log, t_raw, lf_log, ctx)
        total += loss.item()
        for k, v in metrics.items():
            metrics_acc[k] = metrics_acc.get(k, 0.0) + v
        n += 1
    avg = total / max(n, 1)
    for k in metrics_acc: metrics_acc[k] /= max(n, 1)
    return avg, metrics_acc


def train_one_epoch(model, loader, criterion, optimizer, scaler, device, epoch):
    model.train()
    total, metrics_acc, n = 0.0, {}, 0
    for i, (inputs, t_log, t_raw, lf_log, ctx) in enumerate(loader):
        inputs, t_log, t_raw, lf_log, ctx = (
            inputs.to(device, non_blocking=True),
            t_log.to(device, non_blocking=True),
            t_raw.to(device, non_blocking=True),
            lf_log.to(device, non_blocking=True),
            ctx.to(device, non_blocking=True),
        )
        optimizer.zero_grad(set_to_none=True)
        with autocast(dtype=torch.float16):
            pred_log = model(inputs, lf_log)
            loss, metrics = criterion(pred_log.float(), t_log, t_raw, lf_log, ctx)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer); scaler.update()

        total += loss.item()
        for k, v in metrics.items():
            metrics_acc[k] = metrics_acc.get(k, 0.0) + v
        n += 1
        if i % 50 == 0:
            print(f"  Ep{epoch} [{i}/{len(loader)}] "
                  f"loss={loss.item():.4f}  "
                  f"huber={metrics.get('base_loss',0):.4f}  "
                  f"pinball={metrics.get('pinball_loss',0):.4f}  "
                  f"ep_smape={metrics.get('episode_smape',0):.4f}")
    avg = total / max(n, 1)
    for k in metrics_acc: metrics_acc[k] /= max(n, 1)
    return avg, metrics_acc


# ============================================================================
# INFERENCE WITH TEMPORAL PERSISTENCE CALIBRATION
# ============================================================================

@torch.no_grad()
def run_inference_tpc(model, norm_stats, device):
    """
    OOM-safe inference in batches of 4.

    Post-processing order (mandatory):
      1. expm1  → raw µg/m³ space
      2. Temporal Persistence Calibration  → counteract ConvGRU decay
      3. Save preds.npy
    """
    model.eval()
    cfg = Config; ft = FinetuneConfig
    output_path = os.path.join(ft.WORKING_DIR, 'preds.npy')

    # Accumulator: (N, H, W, T)
    preds = np.zeros((cfg.NUM_TEST_SAMPLES, cfg.H, cfg.W, cfg.OUTPUT_HOURS), dtype=np.float32)

    test_ds     = PM25TestDataset(cfg.TEST_DIR, cfg.FEATURES, norm_stats)
    test_loader = DataLoader(test_ds, batch_size=cfg.INFERENCE_BATCH_SIZE,
                             shuffle=False, num_workers=0, pin_memory=True)
    idx = 0
    for batch_x, batch_lf in test_loader:
        batch_x  = batch_x.to(device, non_blocking=True)
        batch_lf = batch_lf.to(device, non_blocking=True)
        bs = batch_x.shape[0]

        with autocast(dtype=torch.float16):
            pred_log = model(batch_x, batch_lf).float()  # (B, 16, H, W)

        # Step 1 — expm1 → raw µg/m³
        pred_raw = F.relu(torch.expm1(pred_log.clamp(max=15)))

        # Permute to (B, H, W, T) before storing
        pred_np = pred_raw.permute(0, 2, 3, 1).cpu().numpy()
        preds[idx:idx+bs] = pred_np
        idx += bs

        if idx % 50 == 0 or idx >= cfg.NUM_TEST_SAMPLES:
            print(f"  Inference: {idx}/{cfg.NUM_TEST_SAMPLES}")

        del batch_x, batch_lf, pred_log, pred_raw
        torch.cuda.empty_cache()

    # Step 2 — Temporal Persistence Calibration
    # The ConvGRU naturally decays intensity for distant time steps.
    # We counteract this with a linear ramp: +0.5 % per output hour.
    # t=0: ×1.000  (no change)
    # t=7: ×1.035  (+3.5 %)
    # t=15: ×1.075 (+7.5 %)
    print("\nApplying Temporal Persistence Calibration (+0.5 % / hour) ...")
    for t in range(cfg.OUTPUT_HOURS):
        scale = 1.0 + ft.TPC_RATE * t
        preds[:, :, :, t] *= scale
        if t in (0, 7, 15):
            print(f"  t={t:2d}: ×{scale:.4f}")

    # Step 3 — save
    np.save(output_path, preds)
    print(f"\nSaved: {output_path}  shape={preds.shape}")
    print(f"Range: [{preds.min():.2f}, {preds.max():.2f}]  mean={preds.mean():.3f}")
    return preds


# ============================================================================
# MAIN
# ============================================================================

def main():
    seed_everything()
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Device: {device}")
    if torch.cuda.is_available():
        print(f"GPU: {torch.cuda.get_device_name(0)}")

    cfg = Config; ft = FinetuneConfig

    # ---- Step 1: Load norm stats ----
    print("\n" + "="*60 + "\nSTEP 1: Loading normalization statistics\n" + "="*60)
    norm_path  = ft.find_file('norm_stats_resgru.npz')
    norm_stats = NormStats()
    if os.path.exists(norm_path):
        print(f"  Loading from {norm_path}")
        norm_stats.load(norm_path)
    else:
        print("  No cache found — recomputing ...")
        norm_stats.compute_from_raw(cfg.RAW_DIR, cfg.MONTHS, cfg.FEATURES)
        norm_stats.save(os.path.join(ft.WORKING_DIR, 'norm_stats_resgru.npz'))
    gc.collect()

    # ---- Step 2: Datasets with STRIDE=1 ----
    print("\n" + "="*60 + f"\nSTEP 2: Creating datasets (STRIDE={ft.STRIDE})\n" + "="*60)
    full_ds = PM25TrainDataset(cfg.RAW_DIR, cfg.MONTHS, cfg.FEATURES, norm_stats, stride=ft.STRIDE)
    n_val   = int(len(full_ds) * ft.VAL_SPLIT)
    n_train = len(full_ds) - n_val
    train_ds, val_ds = torch.utils.data.random_split(
        full_ds, [n_train, n_val],
        generator=torch.Generator().manual_seed(cfg.SEED)
    )
    print(f"  Train: {n_train}  Val: {n_val}")

    train_loader = DataLoader(train_ds, batch_size=ft.BATCH_SIZE, shuffle=True,
                              num_workers=2, pin_memory=True, drop_last=True,
                              persistent_workers=True)
    val_loader   = DataLoader(val_ds,   batch_size=ft.BATCH_SIZE, shuffle=False,
                              num_workers=2, pin_memory=True,
                              persistent_workers=True)

    # ---- Step 3: Load 0.8768 checkpoint ----
    print("\n" + "="*60 + "\nSTEP 3: Loading Phase 2 checkpoint (0.8768)\n" + "="*60)
    base_ckpt_path = ft.find_file('best_episodic_finetune.pt')
    print(f"  Using Phase 2 checkpoint: {base_ckpt_path}")

    assert os.path.exists(base_ckpt_path), (
        f"Checkpoint not found: {base_ckpt_path}\n"
        "On Kaggle: ensure Phase 2 ran successfully, or add the base notebook as a data source."
    )

    model = ResGRUUNet(cfg).to(device)
    ckpt  = torch.load(base_ckpt_path, map_location=device, weights_only=True)
    model.load_state_dict(ckpt['model_state_dict'])
    print(f"  epoch={ckpt.get('epoch','?')}  "
          f"ep_smape={ckpt.get('episode_smape', ckpt.get('val_loss','?'))}")

    # ---- Step 4: Quantile fine-tuning ----
    print("\n" + "="*60 +
          f"\nSTEP 4: Quantile fine-tuning  "
          f"q={ft.QUANTILE}  LR={ft.LR}  Epochs={ft.EPOCHS}\n" + "="*60)
    print(f"  Loss = {ft.HUBER_WEIGHT} × Huber + {ft.PINBALL_WEIGHT} × Pinball(q={ft.QUANTILE})")

    criterion = HuberPinballLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=ft.LR, weight_decay=ft.WEIGHT_DECAY)
    scaler    = GradScaler()

    best_ep_smape = float('inf')
    best_epoch    = 0

    for epoch in range(1, ft.EPOCHS + 1):
        t0 = time.time()
        tr_loss, tr_m = train_one_epoch(model, train_loader, criterion, optimizer, scaler, device, epoch)
        vl_loss, vl_m = validate_quantile(model, val_loader, criterion, device)
        ep_smape = vl_m.get('episode_smape', float('inf'))
        ep_corr  = vl_m.get('episode_corr',  0.0)

        print(f"\nEpoch {epoch}/{ft.EPOCHS}  ({time.time()-t0:.0f}s)")
        print(f"  Train | loss={tr_loss:.4f}  "
              f"huber={tr_m.get('base_loss',0):.4f}  "
              f"pinball={tr_m.get('pinball_loss',0):.4f}  "
              f"ep_smape={tr_m.get('episode_smape',0):.4f}  "
              f"ep_corr={tr_m.get('episode_corr',0):.4f}")
        print(f"  Val   | loss={vl_loss:.4f}  "
              f"global={vl_m.get('global_smape',0):.4f}  "
              f"ep_smape={ep_smape:.4f}  "
              f"ep_corr={ep_corr:.4f}  "
              f"n_ep={vl_m.get('n_episodes',0):.0f}")

        if ep_smape < best_ep_smape:
            best_ep_smape = ep_smape
            best_epoch    = epoch
            torch.save({
                'epoch':          epoch,
                'model_state_dict': model.state_dict(),
                'val_loss':       vl_loss,
                'episode_smape':  ep_smape,
                'episode_corr':   ep_corr,
                'val_metrics':    vl_m,
            }, ft.QUANTILE_CKPT)
            print(f"  -> Saved best  episode_smape={ep_smape:.4f}  ep_corr={ep_corr:.4f}")
        else:
            print(f"  -> No improvement (best={best_ep_smape:.4f} @ epoch {best_epoch})")

    print(f"\nQuantile fine-tuning complete.  Best Episode SMAPE: {best_ep_smape:.4f} @ epoch {best_epoch}")

    # ---- Step 5: OOM-safe inference + Temporal Persistence Calibration ----
    print("\n" + "="*60 + "\nSTEP 5: OOM-safe inference + Temporal Persistence Calibration\n" + "="*60)
    best = torch.load(ft.QUANTILE_CKPT, map_location=device, weights_only=True)
    model.load_state_dict(best['model_state_dict'])
    print(f"  Loaded best quantile checkpoint: epoch {best['epoch']}  "
          f"episode_smape={best['episode_smape']:.4f}")

    del train_loader, val_loader, train_ds, val_ds, full_ds
    gc.collect(); torch.cuda.empty_cache()

    preds = run_inference_tpc(model, norm_stats, device)

    print("\n" + "="*60)
    print("PHASE 3 QUANTILE REFINEMENT COMPLETE")
    print(f"  Checkpoint  : {ft.QUANTILE_CKPT}")
    print(f"  Predictions : {os.path.join(ft.WORKING_DIR, 'preds.npy')}")
    print(f"  Best Ep SMAPE: {best_ep_smape:.4f} @ epoch {best_epoch}")
    print("="*60)


if __name__ == '__main__':
    main()


Device: cuda
GPU: Tesla T4

STEP 1: Loading normalization statistics
  No cache found — recomputing ...
Computing normalization statistics...
  cpm25: done
  q2: done
  t2: done
  u10: done
  v10: done
  pblh: done
  psfc: done
  swdown: done
  rain: done
  PM25: done
  NOx: done
  SO2: done
  NH3: done

STEP 2: Creating datasets (STRIDE=1)
  TrainDataset: 2832 samples, stride=1
  Train: 2549  Val: 283

STEP 3: Loading Phase 2 checkpoint (0.8768)
  Using Phase 2 checkpoint: /kaggle/input/notebooks/kaori02/res-1fine/best_episodic_finetune.pt
  epoch=7  ep_smape=0.12272685312157783

STEP 4: Quantile fine-tuning  q=0.75  LR=2e-05  Epochs=5
  Loss = 0.4 × Huber + 0.6 × Pinball(q=0.75)


/tmp/ipykernel_24/2744710471.py:658: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler    = GradScaler()
/tmp/ipykernel_24/2744710471.py:500: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(dtype=torch.float16):


  Ep1 [0/637] loss=0.0397  huber=0.0218  pinball=0.0517  ep_smape=0.1233
  Ep1 [50/637] loss=0.0368  huber=0.0221  pinball=0.0466  ep_smape=0.0908
  Ep1 [100/637] loss=0.0317  huber=0.0175  pinball=0.0411  ep_smape=0.0901
  Ep1 [150/637] loss=0.0367  huber=0.0207  pinball=0.0475  ep_smape=0.0915
  Ep1 [200/637] loss=0.0312  huber=0.0172  pinball=0.0405  ep_smape=0.0957
  Ep1 [250/637] loss=0.0366  huber=0.0214  pinball=0.0468  ep_smape=0.1036
  Ep1 [300/637] loss=0.0395  huber=0.0239  pinball=0.0498  ep_smape=0.1072
  Ep1 [350/637] loss=0.0376  huber=0.0208  pinball=0.0488  ep_smape=0.1042
  Ep1 [400/637] loss=0.0367  huber=0.0222  pinball=0.0464  ep_smape=0.0962
  Ep1 [450/637] loss=0.0363  huber=0.0211  pinball=0.0464  ep_smape=0.0978
  Ep1 [500/637] loss=0.0329  huber=0.0177  pinball=0.0431  ep_smape=0.0891
  Ep1 [550/637] loss=0.0378  huber=0.0216  pinball=0.0486  ep_smape=0.1078
  Ep1 [600/637] loss=0.0347  huber=0.0183  pinball=0.0456  ep_smape=0.0998


/tmp/ipykernel_24/2744710471.py:476: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(dtype=torch.float16):



Epoch 1/5  (236s)
  Train | loss=0.0364  huber=0.0210  pinball=0.0467  ep_smape=0.0983  ep_corr=0.9865
  Val   | loss=0.0340  global=0.1465  ep_smape=0.0940  ep_corr=0.9875  n_ep=290343
  -> Saved best  episode_smape=0.0940  ep_corr=0.9875
  Ep2 [0/637] loss=0.0343  huber=0.0185  pinball=0.0447  ep_smape=0.1027
  Ep2 [50/637] loss=0.0343  huber=0.0188  pinball=0.0447  ep_smape=0.0954
  Ep2 [100/637] loss=0.0401  huber=0.0252  pinball=0.0501  ep_smape=0.1034
  Ep2 [150/637] loss=0.0358  huber=0.0201  pinball=0.0463  ep_smape=0.1039
  Ep2 [200/637] loss=0.0323  huber=0.0170  pinball=0.0426  ep_smape=0.0912
  Ep2 [250/637] loss=0.0355  huber=0.0208  pinball=0.0453  ep_smape=0.1018
  Ep2 [300/637] loss=0.0388  huber=0.0237  pinball=0.0488  ep_smape=0.0971
  Ep2 [350/637] loss=0.0355  huber=0.0201  pinball=0.0458  ep_smape=0.1050
  Ep2 [400/637] loss=0.0362  huber=0.0196  pinball=0.0473  ep_smape=0.1012
  Ep2 [450/637] loss=0.0384  huber=0.0222  pinball=0.0492  ep_smape=0.1025
  Ep2 [500/6

/tmp/ipykernel_24/2744710471.py:553: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(dtype=torch.float16):


  Inference: 100/218
  Inference: 200/218
  Inference: 218/218

Applying Temporal Persistence Calibration (+0.5 % / hour) ...
  t= 0: ×1.0000
  t= 7: ×1.0350
  t=15: ×1.0750

Saved: /kaggle/working/preds.npy  shape=(218, 140, 124, 16)
Range: [0.00, 1348.14]  mean=40.786

PHASE 3 QUANTILE REFINEMENT COMPLETE
  Checkpoint  : /kaggle/working/best_quantile_finetune.pt
  Predictions : /kaggle/working/preds.npy
  Best Ep SMAPE: 0.0930 @ epoch 2
